### Notebook Setup
As I worked in Colab for this project, I had a few issues working with the datasets and their paths.My first few cells are just for organization and setup. 


In [1]:
from pathlib import Path
import os
import shutil

current_directory = Path.cwd()
if (current_directory / "BestModel").exists() and (current_directory / "data").exists():
    project_root_directory = current_directory
else:
    project_root_directory = Path("/content/NLP_Coursework3")

project_root_directory.mkdir(parents=True, exist_ok=True)
os.chdir(project_root_directory)

raw_data_directory = project_root_directory / "data" / "raw"
raw_data_directory.mkdir(parents=True, exist_ok=True)
(project_root_directory / "data" / "processed").mkdir(parents=True, exist_ok=True)
(project_root_directory / "BestModel").mkdir(parents=True, exist_ok=True)
(project_root_directory / "artifacts" / "models").mkdir(parents=True, exist_ok=True)
(project_root_directory / "artifacts" / "preds").mkdir(parents=True, exist_ok=True)

required_files = [
    "dontpatronizeme_pcl.tsv",
    "dontpatronizeme_categories.tsv",
    "train_semeval_parids-labels.csv",
    "dev_semeval_parids-labels.csv",
]

known_test_file_names = [
    "official_test_set.tsv",
    "test_set.tsv",
    "task4_test.tsv",
    "pcl_test.tsv",
    "test.tsv",
]

upload_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

ignore_names = {
    "dontpatronizeme_pcl.tsv",
    "dontpatronizeme_categories.tsv",
    "train_semeval_parids-labels.csv",
    "dev_semeval_parids-labels.csv",
    "additional_context.txt",
}


def find_candidates(exact_name=None, pattern=None):
    results = []
    for root in upload_roots:
        if not root.exists():
            continue
        iterator = root.rglob(exact_name) if exact_name is not None else root.rglob(pattern)
        for path in iterator:
            if path.is_file():
                results.append(path)
    return results


def copy_latest_into_raw(candidates, destination_name):
    if not candidates:
        return None

    external_candidates = [path for path in candidates if raw_data_directory not in path.parents]
    candidate_pool = external_candidates if external_candidates else candidates
    source_path = max(candidate_pool, key=lambda path: path.stat().st_mtime)

    destination_path = raw_data_directory / destination_name
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        if source_path.resolve() == destination_path.resolve():
            return source_path, destination_path
    except FileNotFoundError:
        pass

    shutil.copy2(source_path, destination_path)
    return source_path, destination_path



moved_or_copied = []

for file_name in required_files:
    result = copy_latest_into_raw(find_candidates(exact_name=file_name), file_name)
    if result is not None:
        source_path, destination_path = result
        moved_or_copied.append((source_path, destination_path))

test_candidates = []
for file_name in known_test_file_names:
    test_candidates.extend(find_candidates(exact_name=file_name))

if not test_candidates:
    for pattern in ["*test*.tsv"]:
        for candidate_path in find_candidates(pattern=pattern):
            lowered_name = candidate_path.name.lower()
            if candidate_path.name in ignore_names:
                continue
            if "train" in lowered_name or "dev" in lowered_name or "categor" in lowered_name:
                continue
            test_candidates.append(candidate_path)

if test_candidates:
    source_path, detected_destination = copy_latest_into_raw(test_candidates, max(test_candidates, key=lambda p: p.stat().st_mtime).name)
    moved_or_copied.append((source_path, detected_destination))

    canonical_test_path = raw_data_directory / "official_test_set.tsv"
    if detected_destination.name != "official_test_set.tsv":
        shutil.copy2(detected_destination, canonical_test_path)
    print(f"Detected test file: {detected_destination.name}")
    print(f"Canonical test alias: {canonical_test_path.name}")
else:
    print("No test input file detected automatically. If needed, rename your file to official_test_set.tsv and rerun this cell.")

print("Project root:", project_root_directory)
for source_path, destination_path in moved_or_copied:
    print(f"Copied: {source_path} -> {destination_path}")

missing_required_files = [name for name in required_files if not (raw_data_directory / name).exists()]
if missing_required_files:
    print("Missing required files:")
    for file_name in missing_required_files:
        print(" -", file_name)
else:
    print("All required files are present in data/raw.")

print("Current data/raw files:")
for path in sorted(raw_data_directory.glob("*")):
    print(" -", path.name)


No test input file detected automatically. If needed, rename your file to official_test_set.tsv and rerun this cell.
Project root: \content\NLP_Coursework3
Missing required files:
 - dontpatronizeme_pcl.tsv
 - dontpatronizeme_categories.tsv
 - train_semeval_parids-labels.csv
 - dev_semeval_parids-labels.csv
Current data/raw files:


### Dependency Setup
These were the libraries I used for the transformer workflow.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

marker_directory = Path('.runtime_markers')
marker_directory.mkdir(parents=True, exist_ok=True)
dependency_marker = marker_directory / 'bestmodel3_hf_dependencies_ready.txt'

if dependency_marker.exists():
    print('Dependencies are already prepared in this runtime. Continue to the next cell.')
else:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-U',
        'transformers',
        'datasets',
        'accelerate',
        'evaluate',
        'sentencepiece',
        'huggingface_hub',
    ])

    dependency_marker.write_text('ready', encoding='utf-8')
    print('Transformer dependencies installed. Restarting runtime now; then rerun from Cell 1.')
    os.kill(os.getpid(), 9)


### Runtime Imports


In [ ]:
import gc
import inspect
import os
from pathlib import Path

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import numpy as np
import pandas as pd
import torch
import sklearn
import transformers
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

raw_data_directory = Path("data/raw")
artifact_directory = Path("artifacts")
best_model_output_directory = Path("BestModel")
(artifact_directory / "models").mkdir(parents=True, exist_ok=True)
(artifact_directory / "preds").mkdir(parents=True, exist_ok=True)
best_model_output_directory.mkdir(parents=True, exist_ok=True)

print("Python package versions")
print(" - torch:", torch.__version__)
print(" - transformers:", transformers.__version__)
print(" - sklearn:", sklearn.__version__)
print(" - numpy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


Python package versions
 - torch: 2.9.0+cu128
 - transformers: 5.1.0
 - sklearn: 1.6.1
 - numpy: 2.0.2
CUDA available: True
CUDA device: NVIDIA A100-SXM4-80GB


### Experiment Configuration
These were my run settings, and I kept them here so the behaviour stayed deterministic.


In [ ]:
run_baseline_experiment = True
official_baseline_development_f1 = 0.48
official_baseline_test_f1 = 0.49

use_text_only_input = True
run_optional_deberta_candidates = False
retrain_on_train_plus_development_for_test = True
internal_validation_fraction_for_retraining = 0.10
minimum_quality_threshold_for_pruned_ensemble = 0.61
minimum_pool_size_for_ensemble = 2
retrain_final_selection_for_test_prediction = True
maximum_runs_to_retrain_for_test = 3

baseline_model_settings = {
    "run_name": "roberta_base_baseline",
    "model_name": "roberta-base",
    "seed": 42,
    "learning_rate": 2e-5,
    "epochs": 3,
    "batch_size": 16,
    "gradient_accumulation_steps": 1,
    "max_length": 192,
    "weight_decay": 0.01,
}

candidate_model_runs = [
    {
        "run_name": "roberta_large_seed_42",
        "model_name": "roberta-large",
        "seed": 42,
        "learning_rate": 1.2e-5,
        "epochs": 4,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "max_length": 192,
        "weight_decay": 0.01,
    },
    {
        "run_name": "roberta_large_seed_3407",
        "model_name": "roberta-large",
        "seed": 3407,
        "learning_rate": 1.2e-5,
        "epochs": 4,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "max_length": 192,
        "weight_decay": 0.01,
    },
]

if run_optional_deberta_candidates:
    candidate_model_runs.extend(
        [
            {
                "run_name": "deberta_v3_large_seed_42",
                "model_name": "microsoft/deberta-v3-large",
                "seed": 42,
                "learning_rate": 1.2e-5,
                "epochs": 3,
                "batch_size": 8,
                "gradient_accumulation_steps": 2,
                "max_length": 192,
                "weight_decay": 0.01,
            },
            {
                "run_name": "deberta_v3_large_seed_3407",
                "model_name": "microsoft/deberta-v3-large",
                "seed": 3407,
                "learning_rate": 1.2e-5,
                "epochs": 3,
                "batch_size": 8,
                "gradient_accumulation_steps": 2,
                "max_length": 192,
                "weight_decay": 0.01,
            },
        ]
    )

print("Baseline run enabled:", run_baseline_experiment)
print("Text-only transformer input:", use_text_only_input)
print("Optional DeBERTa enabled:", run_optional_deberta_candidates)
print("Initial candidate run count:", len(candidate_model_runs))


Baseline run enabled: True
Text-only transformer input: True
Optional DeBERTa enabled: False
Initial candidate run count: 2


### Paragraph ID Parsing


In [ ]:
def parse_integer_identifier(raw_identifier):
    cleaned_identifier = str(raw_identifier).replace("@@", "").strip()
    if cleaned_identifier.isdigit():
        return int(cleaned_identifier)
    return None


### Labelled Dataset Loader


In [ ]:
def load_labelled_pcl_dataset(dataset_path: Path) -> pd.DataFrame:
    parsed_rows = []
    with dataset_path.open("r", encoding="utf-8", errors="replace") as file_handle:
        for text_line in file_handle:
            if text_line.startswith("---") or text_line.startswith("The Don"):
                continue

            cleaned_line = text_line.rstrip("\n")
            if not cleaned_line:
                continue

            columns = cleaned_line.split("	")
            if len(columns) != 6:
                continue

            paragraph_id, article_id, keyword, country_code, paragraph_text, ordinal_label = columns
            parsed_paragraph_id = parse_integer_identifier(paragraph_id)
            if parsed_paragraph_id is None:
                continue

            parsed_rows.append(
                {
                    "par_id": parsed_paragraph_id,
                    "art_id": str(article_id),
                    "keyword": str(keyword),
                    "country": str(country_code),
                    "text": str(paragraph_text),
                    "label_ordinal": int(ordinal_label),
                }
            )

    labelled_dataframe = pd.DataFrame(parsed_rows)
    labelled_dataframe["label_bin"] = (labelled_dataframe["label_ordinal"] >= 2).astype(int)
    return labelled_dataframe


### Unlabelled Test Parser


In [ ]:
def load_unlabelled_test_dataset(test_dataset_path: Path) -> pd.DataFrame:
    import csv

    parsed_rows = []
    with test_dataset_path.open("r", encoding="utf-8", errors="replace") as file_handle:
        raw_lines = [
            line.rstrip("\n")
            for line in file_handle
            if line.strip() and not line.startswith("---") and not line.startswith("The Don")
        ]

    if not raw_lines:
        return pd.DataFrame(columns=["par_id", "art_id", "keyword", "country", "text"])

    sample_lines = raw_lines[:200]
    tab_score = sum(line.count("\t") for line in sample_lines)
    comma_score = sum(line.count(",") for line in sample_lines)

    if tab_score > 0 and tab_score >= comma_score:
        delimiter = "\t"
        split_rows = [line.split("\t") for line in raw_lines]
    elif comma_score > 0:
        delimiter = ","
        split_rows = list(csv.reader(raw_lines))
    else:
        delimiter = " "
        split_rows = [[line] for line in raw_lines]

    if split_rows:
        first_cell = str(split_rows[0][0]).strip().lower()
        if first_cell in {"par_id", "id", "paragraph_id"}:
            split_rows = split_rows[1:]

    for row_index, columns in enumerate(split_rows, start=1):
        columns = [str(value).strip() for value in columns]
        if not columns:
            continue

        parsed_paragraph_id = parse_integer_identifier(columns[0])
        if parsed_paragraph_id is None:
            parsed_paragraph_id = row_index

        article_id = columns[1] if len(columns) > 1 else ""
        keyword = columns[2] if len(columns) > 2 else ""
        country_code = columns[3] if len(columns) > 3 else ""

        if len(columns) >= 5:
            paragraph_text = delimiter.join(columns[4:]).strip()
        else:
            paragraph_text = max(columns, key=len).strip()

        if paragraph_text.lower() in {"text", "paragraph", "sentence"} and len(columns) >= 6:
            paragraph_text = delimiter.join(columns[5:]).strip()

        parsed_rows.append(
            {
                "par_id": int(parsed_paragraph_id),
                "art_id": article_id,
                "keyword": keyword,
                "country": country_code,
                "text": paragraph_text,
            }
        )

    return pd.DataFrame(parsed_rows)


### Model Input Text Builder


In [ ]:
def build_model_input_text(input_dataframe: pd.DataFrame, text_only: bool = True) -> pd.Series:
    text_series = input_dataframe["text"].fillna("").astype(str)
    if text_only:
        return text_series

    keyword_series = input_dataframe["keyword"].fillna("").astype(str)
    country_series = input_dataframe["country"].fillna("").astype(str)
    return "Keyword: " + keyword_series + " [SEP] Country: " + country_series + " [SEP] Paragraph: " + text_series


### Train Dev Test Assembly


In [ ]:
full_dataset_dataframe = load_labelled_pcl_dataset(raw_data_directory / "dontpatronizeme_pcl.tsv")
train_split_identifiers = pd.read_csv(raw_data_directory / "train_semeval_parids-labels.csv")
development_split_identifiers = pd.read_csv(raw_data_directory / "dev_semeval_parids-labels.csv")

train_dataframe = train_split_identifiers[["par_id"]].merge(full_dataset_dataframe, on="par_id", how="left", validate="one_to_one")
development_dataframe = development_split_identifiers[["par_id"]].merge(full_dataset_dataframe, on="par_id", how="left", validate="one_to_one")

if train_dataframe["text"].isna().any() or development_dataframe["text"].isna().any():
    missing_train = int(train_dataframe["text"].isna().sum())
    missing_development = int(development_dataframe["text"].isna().sum())
    raise ValueError(f"Missing rows after split merge. train={missing_train}, dev={missing_development}")

train_dataframe = train_dataframe.copy()
development_dataframe = development_dataframe.copy()
train_dataframe["model_input_text"] = build_model_input_text(train_dataframe, text_only=use_text_only_input)
development_dataframe["model_input_text"] = build_model_input_text(development_dataframe, text_only=use_text_only_input)

official_test_file_path = find_official_test_file(raw_data_directory)

test_dataframe = load_unlabelled_test_dataset(official_test_file_path)
if test_dataframe.empty:
    raise ValueError(f"Found a test file but parsing failed: {official_test_file_path}")
if len(test_dataframe) != 3832:
    raise AssertionError(f"Expected 3832 rows in official test set, found {len(test_dataframe)} from {official_test_file_path}")

test_dataframe["model_input_text"] = build_model_input_text(test_dataframe, text_only=use_text_only_input)

print(f"Test file detected: {official_test_file_path.name} | Rows: {len(test_dataframe):,}")
print(f"Train rows: {len(train_dataframe):,} | Positive rate: {train_dataframe['label_bin'].mean():.4f}")
print(f"Dev rows:   {len(development_dataframe):,} | Positive rate: {development_dataframe['label_bin'].mean():.4f}")


Test file detected: task4_test.tsv | Rows: 3,832
Train rows: 8,375 | Positive rate: 0.0948
Dev rows:   2,094 | Positive rate: 0.0950


### Logits to Probabilities Helper
As thresholding and ensembling needed calibrated positive class probabilities, I added this helper to convert logits to probabilities for all my runs.


In [ ]:
def convert_logits_to_probabilities(logits: np.ndarray) -> np.ndarray:
    stabilized_logits = logits - logits.max(axis=1, keepdims=True)
    exponentiated_logits = np.exp(stabilized_logits)
    return exponentiated_logits / exponentiated_logits.sum(axis=1, keepdims=True)


### Threshold Optimisation Helper
I added this helper to search for the threshold that gave the best positive class F1 on the dev set.


In [ ]:
def find_best_threshold_for_positive_f1(true_labels: np.ndarray, positive_probabilities: np.ndarray):
    best_threshold = 0.5
    best_f1_score = -1.0
    for threshold in np.arange(0.05, 0.951, 0.005):
        predictions = (positive_probabilities >= threshold).astype(int)
        current_f1 = f1_score(true_labels, predictions, pos_label=1, zero_division=0)
        if current_f1 > best_f1_score:
            best_f1_score = float(current_f1)
            best_threshold = float(threshold)
    return best_threshold, best_f1_score


### DataFrame to Hugging Face Dataset Conversion


In [ ]:
def convert_dataframe_to_huggingface_dataset(input_dataframe: pd.DataFrame, include_labels: bool) -> Dataset:
    if include_labels:
        dataset = Dataset.from_pandas(input_dataframe[["model_input_text", "label_bin"]].copy(), preserve_index=False)
        dataset = dataset.rename_column("label_bin", "labels")
    else:
        dataset = Dataset.from_pandas(input_dataframe[["model_input_text"]].copy(), preserve_index=False)
    return dataset


### Trainer Metrics Callback
I added this callback to track precision, recall, and positive class F1 during evaluation.


In [ ]:
def compute_binary_metrics_from_logits(eval_prediction):
    logits, labels = eval_prediction
    probabilities = convert_logits_to_probabilities(logits)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    precision, recall, f1_score_value, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )
    return {
        "precision_pos": float(precision),
        "recall_pos": float(recall),
        "f1_positive_class": float(f1_score_value),
    }


### TrainingArguments Compatibility Builder
As I had issues with different runtime and library versions, I created this wrapper for argument creation so my training configuration stayed stable.


In [ ]:
def build_training_arguments_for_run(model_run_settings: dict) -> TrainingArguments:
    use_cuda = torch.cuda.is_available()
    use_bfloat16 = bool(use_cuda and torch.cuda.is_bf16_supported())

    arguments_signature = inspect.signature(TrainingArguments.__init__).parameters
    training_argument_kwargs = {
        "output_dir": str(artifact_directory / "models" / model_run_settings["run_name"]),
        "learning_rate": model_run_settings["learning_rate"],
        "per_device_train_batch_size": model_run_settings["batch_size"],
        "per_device_eval_batch_size": model_run_settings["batch_size"] * 2,
        "gradient_accumulation_steps": model_run_settings["gradient_accumulation_steps"],
        "num_train_epochs": model_run_settings["epochs"],
        "weight_decay": model_run_settings["weight_decay"],
        "lr_scheduler_type": "cosine",
        "logging_steps": 50,
        "report_to": "none",
        "seed": model_run_settings["seed"],
        "dataloader_num_workers": 2 if use_cuda else 0,
    }

    if "eval_strategy" in arguments_signature:
        training_argument_kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in arguments_signature:
        training_argument_kwargs["evaluation_strategy"] = "epoch"

    if "save_strategy" in arguments_signature:
        training_argument_kwargs["save_strategy"] = "no"
    if "load_best_model_at_end" in arguments_signature:
        training_argument_kwargs["load_best_model_at_end"] = False
    if "save_safetensors" in arguments_signature:
        training_argument_kwargs["save_safetensors"] = False

    if "warmup_ratio" in arguments_signature:
        training_argument_kwargs["warmup_ratio"] = 0.06
    elif "warmup_steps" in arguments_signature:
        training_argument_kwargs["warmup_steps"] = 0

    if "bf16" in arguments_signature:
        training_argument_kwargs["bf16"] = use_bfloat16
    if "fp16" in arguments_signature:
        training_argument_kwargs["fp16"] = bool(use_cuda and not use_bfloat16)

    return TrainingArguments(**training_argument_kwargs)


### Single Model Run Trainer
This is my main per run training function.


In [ ]:
def train_single_model_run(
    model_run_settings: dict,
    training_dataframe: pd.DataFrame,
    evaluation_dataframe: pd.DataFrame,
    prediction_dataframe: pd.DataFrame | None = None,
):
    set_seed(model_run_settings["seed"])

    tokenizer = AutoTokenizer.from_pretrained(model_run_settings["model_name"], use_fast=True)

    training_dataset = convert_dataframe_to_huggingface_dataset(training_dataframe, include_labels=True)
    evaluation_dataset = convert_dataframe_to_huggingface_dataset(evaluation_dataframe, include_labels=True)
    prediction_dataset = (
        convert_dataframe_to_huggingface_dataset(prediction_dataframe, include_labels=False)
        if prediction_dataframe is not None
        else None
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["model_input_text"],
            truncation=True,
            max_length=model_run_settings["max_length"],
        )

    tokenized_training_dataset = training_dataset.map(tokenize_batch, batched=True, remove_columns=["model_input_text"])
    tokenized_evaluation_dataset = evaluation_dataset.map(tokenize_batch, batched=True, remove_columns=["model_input_text"])
    tokenized_prediction_dataset = (
        prediction_dataset.map(tokenize_batch, batched=True, remove_columns=["model_input_text"])
        if prediction_dataset is not None
        else None
    )

    model = AutoModelForSequenceClassification.from_pretrained(model_run_settings["model_name"], num_labels=2)
    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    training_arguments = build_training_arguments_for_run(model_run_settings)

    trainer_kwargs = {
        "model": model,
        "args": training_arguments,
        "train_dataset": tokenized_training_dataset,
        "eval_dataset": tokenized_evaluation_dataset,
        "data_collator": DataCollatorWithPadding(tokenizer=tokenizer),
        "compute_metrics": compute_binary_metrics_from_logits,
    }

    trainer_signature = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_signature:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_signature:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)
    trainer.train()

    evaluation_logits = trainer.predict(tokenized_evaluation_dataset).predictions
    evaluation_probabilities = convert_logits_to_probabilities(evaluation_logits)[:, 1]

    prediction_probabilities = None
    if tokenized_prediction_dataset is not None:
        prediction_logits = trainer.predict(tokenized_prediction_dataset).predictions
        prediction_probabilities = convert_logits_to_probabilities(prediction_logits)[:, 1]

    final_model_directory = artifact_directory / "models" / model_run_settings["run_name"] / "final"
    final_model_directory.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(final_model_directory), safe_serialization=False)
    tokenizer.save_pretrained(str(final_model_directory))

    del trainer, model, tokenizer
    del training_dataset, evaluation_dataset, prediction_dataset
    del tokenized_training_dataset, tokenized_evaluation_dataset, tokenized_prediction_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return evaluation_probabilities, prediction_probabilities


### Baseline Training and Reporting
Here I tried to reproduce the RoBERTa base baseline first so I could compare later candidate runs against a fixed reference.


In [ ]:
development_true_labels = development_dataframe["label_bin"].astype(int).to_numpy()
baseline_result = None
successful_model_runs = []
failed_model_runs = []

if run_baseline_experiment:
    print("=== Baseline run: roberta-base ===")
    baseline_development_probabilities, baseline_test_probabilities = train_single_model_run(
        baseline_model_settings,
        train_dataframe,
        development_dataframe,
        test_dataframe,
    )

    baseline_predictions_at_default_threshold = (baseline_development_probabilities >= 0.5).astype(int)
    baseline_f1_at_default_threshold = f1_score(
        development_true_labels,
        baseline_predictions_at_default_threshold,
        pos_label=1,
        zero_division=0,
    )

    baseline_best_threshold, baseline_best_f1_score = find_best_threshold_for_positive_f1(
        development_true_labels,
        baseline_development_probabilities,
    )
    baseline_predictions_at_best_threshold = (baseline_development_probabilities >= baseline_best_threshold).astype(int)

    baseline_result = {
        "run_name": baseline_model_settings["run_name"],
        "official_baseline_dev_f1": official_baseline_development_f1,
        "reproduced_f1_at_0_5": float(baseline_f1_at_default_threshold),
        "reproduced_best_dev_f1": float(baseline_best_f1_score),
        "reproduced_best_threshold": float(baseline_best_threshold),
    }

    print(f"Official baseline dev F1 (paper/task): {official_baseline_development_f1:.2f}")
    print(f"Reproduced roberta-base F1 @0.5: {baseline_f1_at_default_threshold:.4f}")
    print(f"Reproduced roberta-base best F1: {baseline_best_f1_score:.4f} @ threshold {baseline_best_threshold:.3f}")
    print(classification_report(development_true_labels, baseline_predictions_at_best_threshold, digits=4, zero_division=0))

    baseline_prediction_path = artifact_directory / "preds" / "baseline_dev.txt"
    baseline_prediction_path.write_text(
        "\n".join(map(str, baseline_predictions_at_best_threshold.tolist())) + "\n",
        encoding="utf-8",
    )
    print(f"Saved baseline dev predictions: {baseline_prediction_path}")


if baseline_result is not None:
    print("\nBaseline summary")
    display(pd.DataFrame([baseline_result]))


=== Baseline run: roberta-base ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision Pos,Recall Pos,F1 Positive Class
1,0.251342,0.233793,0.635593,0.376884,0.473186
2,0.219241,0.232575,0.677165,0.432161,0.527607
3,0.193755,0.299139,0.652174,0.527638,0.583333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Official baseline dev F1 (paper/task): 0.48
Reproduced roberta-base F1 @0.5: 0.5833
Reproduced roberta-base best F1: 0.6101 @ threshold 0.050
              precision    recall  f1-score   support

           0     0.9645    0.9451    0.9547      1895
           1     0.5612    0.6683    0.6101       199

    accuracy                         0.9188      2094
   macro avg     0.7628    0.8067    0.7824      2094
weighted avg     0.9261    0.9188    0.9219      2094

Saved baseline dev predictions: artifacts/preds/baseline_dev.txt

Baseline summary


,run_name,official_baseline_dev_f1,reproduced_f1_at_0_5,reproduced_best_dev_f1,reproduced_best_threshold
0,roberta_base_baseline,0.48,0.583333,0.610092,0.05


### Initial Candidate Training
I trained the initial candidate runs here so I could establish a first ranking and see which direction looked most promising.


In [ ]:
print("\n=== Candidate runs ===")
for model_run_settings in candidate_model_runs:
    print(f"\nTraining {model_run_settings['run_name']} ({model_run_settings['model_name']})")
    try:
        development_probabilities_for_run, test_probabilities_for_run = train_single_model_run(
            model_run_settings,
            train_dataframe,
            development_dataframe,
            test_dataframe,
        )
    except Exception as error:
        error_message = str(error)
        failed_model_runs.append({"run_name": model_run_settings["run_name"], "error": error_message})
        print(f"Run failed and will be skipped: {error_message}")
        continue

    best_threshold, best_f1_score = find_best_threshold_for_positive_f1(
        development_true_labels,
        development_probabilities_for_run,
    )
    successful_model_runs.append(
        {
            "settings": model_run_settings,
            "run_name": model_run_settings["run_name"],
            "dev_probabilities": development_probabilities_for_run,
            "test_probabilities": test_probabilities_for_run,
            "best_threshold": best_threshold,
            "best_dev_f1": best_f1_score,
        }
    )
    print(f"Best dev F1: {best_f1_score:.4f} @ threshold {best_threshold:.3f}")

if failed_model_runs:
    print("\nFailed runs summary")
    display(pd.DataFrame(failed_model_runs))



=== Candidate runs ===

Training roberta_large_seed_42 (roberta-large)


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision Pos,Recall Pos,F1 Positive Class
1,0.446037,0.216001,0.579882,0.492462,0.532609
2,0.435120,0.236576,0.581197,0.683417,0.628176
3,0.204130,0.349638,0.660256,0.517588,0.580282
4,0.069930,0.393284,0.635294,0.542714,0.585366


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best dev F1: 0.6025 @ threshold 0.065


### Additional Sweep Configuration
Here I expanded the sweep with additional fixed seeds and control runs while keeping the same core settings.


In [ ]:
additional_roberta_large_seeds = [7, 11, 21, 84, 123, 2024]
include_roberta_base_control_runs = True
additional_roberta_base_seeds = [7, 2024]

# Build templates from existing run settings so hyperparameters stay consistent.
roberta_large_template = None
for run_settings in candidate_model_runs:
    if run_settings["model_name"] == "roberta-large":
        roberta_large_template = run_settings.copy()
        break
if roberta_large_template is None:
    raise RuntimeError("Could not find a roberta-large template in candidate_model_runs.")

roberta_base_template = baseline_model_settings.copy()

existing_run_names = {run_output["run_name"] for run_output in successful_model_runs}
existing_run_names.update(
    {
        failed_run.get("run_name")
        for failed_run in failed_model_runs
        if isinstance(failed_run, dict)
    }
)

additional_candidate_runs = []
for seed_value in additional_roberta_large_seeds:
    run_name = f"roberta_large_seed_{seed_value}_extra"
    if run_name in existing_run_names:
        continue
    new_settings = roberta_large_template.copy()
    new_settings["seed"] = int(seed_value)
    new_settings["run_name"] = run_name
    additional_candidate_runs.append(new_settings)

if include_roberta_base_control_runs:
    for seed_value in additional_roberta_base_seeds:
        run_name = f"roberta_base_seed_{seed_value}_extra"
        if run_name in existing_run_names:
            continue
        new_settings = roberta_base_template.copy()
        new_settings["seed"] = int(seed_value)
        new_settings["run_name"] = run_name
        additional_candidate_runs.append(new_settings)

print(f"New runs queued: {len(additional_candidate_runs)}")
if additional_candidate_runs:
    display(
        pd.DataFrame(additional_candidate_runs)[
            ["run_name", "model_name", "seed", "learning_rate", "epochs", "batch_size"]
        ]
    )
else:
    print("No new runs were added (all names already present).")


New runs queued: 8


,run_name,model_name,seed,learning_rate,epochs,batch_size
0,roberta_large_seed_7_extra,roberta-large,7,0.000012,4,8
1,roberta_large_seed_11_extra,roberta-large,11,0.000012,4,8
2,roberta_large_seed_21_extra,roberta-large,21,0.000012,4,8
3,roberta_large_seed_84_extra,roberta-large,84,0.000012,4,8
4,roberta_large_seed_123_extra,roberta-large,123,0.000012,4,8
5,roberta_large_seed_2024_extra,roberta-large,2024,0.000012,4,8
6,roberta_base_seed_7_extra,roberta-base,7,0.000020,3,16
7,roberta_base_seed_2024_extra,roberta-base,2024,0.000020,3,16


### Additional Sweep Training
I then ran the additional sweep here and appended the successful runs to the same results pool.


In [ ]:
extended_successful_model_runs = []
extended_failed_model_runs = []

if not additional_candidate_runs:
    print("No additional runs to train.")
else:
    for model_run_settings in additional_candidate_runs:
        print(f"\nTraining {model_run_settings['run_name']} ({model_run_settings['model_name']})")
        try:
            development_probabilities_for_run, test_probabilities_for_run = train_single_model_run(
                model_run_settings,
                train_dataframe,
                development_dataframe,
                test_dataframe,
            )
        except Exception as error:
            error_message = str(error)
            failure_record = {"run_name": model_run_settings["run_name"], "error": error_message}
            extended_failed_model_runs.append(failure_record)
            failed_model_runs.append(failure_record)
            print(f"Run failed and will be skipped: {error_message}")
            continue

        best_threshold, best_f1_score = find_best_threshold_for_positive_f1(
            development_true_labels,
            development_probabilities_for_run,
        )

        run_record = {
            "settings": model_run_settings,
            "run_name": model_run_settings["run_name"],
            "dev_probabilities": development_probabilities_for_run,
            "test_probabilities": test_probabilities_for_run,
            "best_threshold": best_threshold,
            "best_dev_f1": best_f1_score,
        }

        successful_model_runs.append(run_record)
        extended_successful_model_runs.append(run_record)
        print(f"Best dev F1: {best_f1_score:.4f} @ threshold {best_threshold:.3f}")

if extended_successful_model_runs:
    print("\nAdditional successful runs")
    display(
        pd.DataFrame(
            [
                {
                    "run_name": run_output["run_name"],
                    "best_threshold": run_output["best_threshold"],
                    "best_dev_f1": run_output["best_dev_f1"],
                }
                for run_output in extended_successful_model_runs
            ]
        ).sort_values("best_dev_f1", ascending=False)
    )

if extended_failed_model_runs:
    print("\nAdditional failed runs")
    display(pd.DataFrame(extended_failed_model_runs))



Training roberta_large_seed_7_extra (roberta-large)


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision Pos,Recall Pos,F1 Positive Class
1,0.219983,0.216876,0.775862,0.226131,0.350195
2,0.160386,0.214790,0.684211,0.457286,0.548193
3,0.090735,0.288330,0.644970,0.547739,0.592391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best dev F1: 0.6070 @ threshold 0.235

Additional successful runs


,run_name,best_threshold,best_dev_f1
3,roberta_large_seed_84_extra,0.070,0.635897
5,roberta_large_seed_2024_extra,0.070,0.619165
4,roberta_large_seed_123_extra,0.065,0.618926
7,roberta_base_seed_2024_extra,0.235,0.606965
2,roberta_large_seed_21_extra,0.805,0.600567
6,roberta_base_seed_7_extra,0.550,0.598383
1,roberta_large_seed_11_extra,0.050,0.590909
0,roberta_large_seed_7_extra,0.065,0.589147


### Pruned Ensemble Ranking and Evaluation
Then I ranked all my successful runs, built a pruned weighted ensemble from the stronger ones, and compared it against the best single run on dev.


In [ ]:
if not successful_model_runs:
    raise RuntimeError("No successful model runs are available.")

all_run_results_dataframe = pd.DataFrame(
    [
        {
            "run_name": run_output["run_name"],
            "model_name": run_output["settings"]["model_name"],
            "seed": run_output["settings"]["seed"],
            "best_threshold": run_output["best_threshold"],
            "best_dev_f1": run_output["best_dev_f1"],
        }
        for run_output in successful_model_runs
    ]
).sort_values("best_dev_f1", ascending=False).reset_index(drop=True)

print("All successful runs so far")
display(all_run_results_dataframe)

best_single_run_after_sweep = max(successful_model_runs, key=lambda run_output: run_output["best_dev_f1"])
print(
    f"Best single run: {best_single_run_after_sweep['run_name']} | "
    f"dev F1={best_single_run_after_sweep['best_dev_f1']:.4f} | "
    f"threshold={best_single_run_after_sweep['best_threshold']:.3f}"
)

pruned_ensemble_pool = [
    run_output
    for run_output in successful_model_runs
    if run_output["best_dev_f1"] >= minimum_quality_threshold_for_pruned_ensemble
]

if len(pruned_ensemble_pool) < minimum_pool_size_for_ensemble:
    sorted_runs = sorted(successful_model_runs, key=lambda run_output: run_output["best_dev_f1"], reverse=True)
    pruned_ensemble_pool = sorted_runs[:minimum_pool_size_for_ensemble]
    print(
        f"Pruned pool below minimum size; using top {minimum_pool_size_for_ensemble} runs instead."
    )

pruned_pool_dataframe = pd.DataFrame(
    [
        {
            "run_name": run_output["run_name"],
            "best_dev_f1": run_output["best_dev_f1"],
            "best_threshold": run_output["best_threshold"],
        }
        for run_output in pruned_ensemble_pool
    ]
).sort_values("best_dev_f1", ascending=False).reset_index(drop=True)

print("Pruned ensemble pool")
display(pruned_pool_dataframe)

pruned_pool_weights = np.array(
    [max(1e-6, run_output["best_dev_f1"]) for run_output in pruned_ensemble_pool],
    dtype=float,
)
pruned_pool_weights = pruned_pool_weights / pruned_pool_weights.sum()

pruned_ensemble_development_probabilities = np.average(
    np.vstack([run_output["dev_probabilities"] for run_output in pruned_ensemble_pool]),
    axis=0,
    weights=pruned_pool_weights,
)
pruned_ensemble_threshold, pruned_ensemble_f1 = find_best_threshold_for_positive_f1(
    development_true_labels,
    pruned_ensemble_development_probabilities,
)
pruned_ensemble_development_predictions = (
    pruned_ensemble_development_probabilities >= pruned_ensemble_threshold
).astype(int)

print(
    f"Pruned ensemble dev F1: {pruned_ensemble_f1:.4f} @ threshold {pruned_ensemble_threshold:.3f}"
)
print(
    classification_report(
        development_true_labels,
        pruned_ensemble_development_predictions,
        digits=4,
        zero_division=0,
    )
)
print("Confusion matrix:")
print(confusion_matrix(development_true_labels, pruned_ensemble_development_predictions))


All successful runs so far


,run_name,model_name,seed,best_threshold,best_dev_f1
0,roberta_large_seed_84_extra,roberta-large,84,0.070,0.635897
1,roberta_large_seed_42,roberta-large,42,0.285,0.621410
2,roberta_large_seed_2024_extra,roberta-large,2024,0.070,0.619165
3,roberta_large_seed_123_extra,roberta-large,123,0.065,0.618926
4,roberta_base_seed_2024_extra,roberta-base,2024,0.235,0.606965
5,roberta_large_seed_3407,roberta-large,3407,0.065,0.602469
6,roberta_large_seed_21_extra,roberta-large,21,0.805,0.600567
7,roberta_base_seed_7_extra,roberta-base,7,0.550,0.598383
8,roberta_large_seed_11_extra,roberta-large,11,0.050,0.590909
9,roberta_large_seed_7_extra,roberta-large,7,0.065,0.589147


Best single run: roberta_large_seed_84_extra | dev F1=0.6359 | threshold=0.070
Pruned ensemble pool


,run_name,best_dev_f1,best_threshold
0,roberta_large_seed_84_extra,0.635897,0.070
1,roberta_large_seed_42,0.621410,0.285
2,roberta_large_seed_2024_extra,0.619165,0.070
3,roberta_large_seed_123_extra,0.618926,0.065


Pruned ensemble dev F1: 0.6413 @ threshold 0.080
              precision    recall  f1-score   support

           0     0.9697    0.9451    0.9572      1895
           1     0.5789    0.7186    0.6413       199

    accuracy                         0.9236      2094
   macro avg     0.7743    0.8319    0.7992      2094
weighted avg     0.9325    0.9236    0.9272      2094

Confusion matrix:
[[1791  104]
 [  56  143]]


### Final Strategy Selection and Export
I selected the better final strategy, then exported the final `dev.txt` and `test.txt` predictions.


In [ ]:
use_pruned_ensemble = pruned_ensemble_f1 > best_single_run_after_sweep["best_dev_f1"]

if use_pruned_ensemble:
    selected_strategy_name = "pruned_weighted_ensemble"
    final_development_predictions = pruned_ensemble_development_predictions
    final_development_probabilities = pruned_ensemble_development_probabilities
    final_threshold = pruned_ensemble_threshold
    final_dev_f1 = pruned_ensemble_f1
else:
    selected_strategy_name = f"best_single::{best_single_run_after_sweep['run_name']}"
    final_threshold = best_single_run_after_sweep["best_threshold"]
    final_development_probabilities = best_single_run_after_sweep["dev_probabilities"]
    final_development_predictions = (final_development_probabilities >= final_threshold).astype(int)
    final_dev_f1 = best_single_run_after_sweep["best_dev_f1"]

print(f"Selected final strategy: {selected_strategy_name}")
print(f"Final dev F1: {final_dev_f1:.4f} @ threshold {final_threshold:.3f}")

final_dev_path = best_model_output_directory / "dev.txt"
final_dev_path.write_text(
    "\n".join(map(str, final_development_predictions.tolist())) + "\n",
    encoding="utf-8",
)
print(f"Saved final dev predictions: {final_dev_path} ({len(final_development_predictions)} lines)")

if test_dataframe is None:
    print("No official test file detected, so test export is skipped.")
else:
    final_test_probabilities = None

    if use_pruned_ensemble:
        if retrain_final_selection_for_test_prediction:
            runs_for_retraining = sorted(
                pruned_ensemble_pool,
                key=lambda run_output: run_output["best_dev_f1"],
                reverse=True,
            )[:maximum_runs_to_retrain_for_test]

            combined_training_dataframe = pd.concat([train_dataframe, development_dataframe], ignore_index=True)
            retrained_test_probabilities = []
            retrained_run_weights = []

            for run_output in runs_for_retraining:
                base_settings = run_output["settings"]
                training_split_dataframe, validation_split_dataframe = train_test_split(
                    combined_training_dataframe,
                    test_size=internal_validation_fraction_for_retraining,
                    stratify=combined_training_dataframe["label_bin"],
                    random_state=base_settings["seed"],
                )
                training_split_dataframe = training_split_dataframe.reset_index(drop=True)
                validation_split_dataframe = validation_split_dataframe.reset_index(drop=True)

                retrain_settings = base_settings.copy()
                retrain_settings["run_name"] = base_settings["run_name"] + "_final_retrain"

                print(f"Retraining for test export: {retrain_settings['run_name']}")
                _, retrained_test_probabilities_for_run = train_single_model_run(
                    retrain_settings,
                    training_split_dataframe,
                    validation_split_dataframe,
                    test_dataframe,
                )

                retrained_test_probabilities.append(retrained_test_probabilities_for_run)
                retrained_run_weights.append(max(1e-6, run_output["best_dev_f1"]))

            retrained_run_weights = np.array(retrained_run_weights, dtype=float)
            retrained_run_weights = retrained_run_weights / retrained_run_weights.sum()
            final_test_probabilities = np.average(
                np.vstack(retrained_test_probabilities),
                axis=0,
                weights=retrained_run_weights,
            )
        else:
            available_test_records = [
                run_output
                for run_output in pruned_ensemble_pool
                if run_output["test_probabilities"] is not None
            ]
            if not available_test_records:
                raise RuntimeError("Pruned ensemble has no available test probabilities.")

            available_test_weights = np.array(
                [max(1e-6, run_output["best_dev_f1"]) for run_output in available_test_records],
                dtype=float,
            )
            available_test_weights = available_test_weights / available_test_weights.sum()
            final_test_probabilities = np.average(
                np.vstack([run_output["test_probabilities"] for run_output in available_test_records]),
                axis=0,
                weights=available_test_weights,
            )
    else:
        if retrain_final_selection_for_test_prediction:
            combined_training_dataframe = pd.concat([train_dataframe, development_dataframe], ignore_index=True)
            base_settings = best_single_run_after_sweep["settings"]

            training_split_dataframe, validation_split_dataframe = train_test_split(
                combined_training_dataframe,
                test_size=internal_validation_fraction_for_retraining,
                stratify=combined_training_dataframe["label_bin"],
                random_state=base_settings["seed"],
            )
            training_split_dataframe = training_split_dataframe.reset_index(drop=True)
            validation_split_dataframe = validation_split_dataframe.reset_index(drop=True)

            retrain_settings = base_settings.copy()
            retrain_settings["run_name"] = base_settings["run_name"] + "_final_retrain"

            print(f"Retraining best single model for test export: {retrain_settings['run_name']}")
            _, final_test_probabilities = train_single_model_run(
                retrain_settings,
                training_split_dataframe,
                validation_split_dataframe,
                test_dataframe,
            )
        else:
            final_test_probabilities = best_single_run_after_sweep["test_probabilities"]
            if final_test_probabilities is None:
                raise RuntimeError("Best single run has no available test probabilities.")

    final_test_predictions = (final_test_probabilities >= final_threshold).astype(int)
    final_test_path = best_model_output_directory / "test.txt"
    final_test_path.write_text(
        "\n".join(map(str, final_test_predictions.tolist())) + "\n",
        encoding="utf-8",
    )
    print(f"Saved final test predictions: {final_test_path} ({len(final_test_predictions)} lines)")

final_strategy_summary = pd.DataFrame(
    [
        {
            "selected_strategy": selected_strategy_name,
            "selected_dev_f1": float(final_dev_f1),
            "selected_threshold": float(final_threshold),
            "best_single_run": best_single_run_after_sweep["run_name"],
            "best_single_dev_f1": float(best_single_run_after_sweep["best_dev_f1"]),
            "pruned_ensemble_dev_f1": float(pruned_ensemble_f1),
            "pool_size": int(len(pruned_ensemble_pool)),
            "quality_threshold": float(minimum_quality_threshold_for_pruned_ensemble),
        }
    ]
)

summary_path = artifact_directory / "preds" / "final_strategy_summary.csv"
final_strategy_summary.to_csv(summary_path, index=False)
print(f"Saved strategy summary: {summary_path}")
display(final_strategy_summary)


Selected final strategy: pruned_weighted_ensemble
Final dev F1: 0.6413 @ threshold 0.080
Saved final dev predictions: BestModel/dev.txt (2094 lines)
Retraining for test export: roberta_large_seed_84_extra_final_retrain


Map:   0%|          | 0/9422 [00:00<?, ? examples/s]

Map:   0%|          | 0/1047 [00:00<?, ? examples/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision Pos,Recall Pos,F1 Positive Class
1,0.502452,0.197808,0.584071,0.666667,0.622642
2,0.339617,0.208983,0.745763,0.444444,0.556962
3,0.241710,0.342095,0.698630,0.515152,0.593023
4,0.114950,0.375626,0.658824,0.565657,0.608696


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final test predictions: BestModel/test.txt (3832 lines)
Saved strategy summary: artifacts/preds/final_strategy_summary.csv


,selected_strategy,selected_dev_f1,selected_threshold,best_single_run,best_single_dev_f1,pruned_ensemble_dev_f1,pool_size,quality_threshold
0,pruned_weighted_ensemble,0.641256,0.08,roberta_large_seed_84_extra,0.635897,0.641256,4,0.61


### Submission File Validation
I validated line counts and label format here, then recomputed dev F1 from the saved file as a final check.


In [ ]:
expected_dev_lines = len(development_dataframe)
expected_test_lines = len(test_dataframe)

final_dev_path = best_model_output_directory / "dev.txt"
final_test_path = best_model_output_directory / "test.txt"
if not final_dev_path.exists() or not final_test_path.exists():
    raise FileNotFoundError("Missing output files. Expected BestModel/dev.txt and BestModel/test.txt.")

dev_predictions = pd.read_csv(final_dev_path, header=None, names=["pred"])["pred"].astype(int)
test_predictions = pd.read_csv(final_test_path, header=None, names=["pred"])["pred"].astype(int)
if len(dev_predictions) != expected_dev_lines:
    raise AssertionError(f"dev.txt line count mismatch: expected {expected_dev_lines}, got {len(dev_predictions)}")
if len(test_predictions) != expected_test_lines:
    raise AssertionError(f"test.txt line count mismatch: expected {expected_test_lines}, got {len(test_predictions)}")

non_binary_dev = int((~dev_predictions.astype(str).str.fullmatch(r"[01]")).sum())
non_binary_test = int((~test_predictions.astype(str).str.fullmatch(r"[01]")).sum())
if non_binary_dev > 0 or non_binary_test > 0:
    raise AssertionError(f"Non-binary predictions detected. dev_non_binary={non_binary_dev}, test_non_binary={non_binary_test}")

final_dev_f1_from_file = f1_score(development_true_labels, dev_predictions, pos_label=1, zero_division=0)
print("Validation summary")
print(f" - dev.txt lines:  {len(dev_predictions)} (expected {expected_dev_lines})")
print(f" - test.txt lines: {len(test_predictions)} (expected {expected_test_lines})")
print(f" - Final dev F1 from saved dev.txt: {final_dev_f1_from_file:.6f}")
print(f" - Prior reference dev F1:          0.630872")
print(f" - Absolute difference:             {abs(final_dev_f1_from_file - 0.6308724832214765):.6f}")
if baseline_result is not None:
    print("\nBaseline checkpoint")
    print(f" - Official baseline dev F1: {baseline_result['official_baseline_dev_f1']:.2f}")
    print(f" - Reproduced baseline best dev F1: {baseline_result['reproduced_best_dev_f1']:.6f} @ threshold {baseline_result['reproduced_best_threshold']:.3f}")


Validation summary
 - dev.txt lines:  2094 (expected 2094)
 - test.txt lines: 3832 (expected 3832)
 - Final dev F1 from saved dev.txt: 0.641256
 - Prior reference dev F1:          0.630872
 - Absolute difference:             0.010383

Baseline checkpoint
 - Official baseline dev F1: 0.48
 - Reproduced baseline best dev F1: 0.610092 @ threshold 0.050
